In [1]:
import glob, os
DATA  = "/kaggle/input/datasets/karrychuong/qudar-beir-data/data"
PATCH = "/kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches"
RUNS  = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/fiqa_test/retrieval_result.json", recursive=True)[0]))
print("RUNS =", RUNS, os.listdir(RUNS))

RUNS = /kaggle/input/datasets/karrychuong/qudar-runs/runs ['climate-fever_test', 'fiqa_test', 'fiqa_train', 'hotpotqa_test', 'hotpotqa_train', 'scidocs_test']


In [2]:
from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
!pip install -q -e /kaggle/working/QuDAR
L = f"python {PATCH}/llm_score.py --repo /kaggle/working/QuDAR --runs {RUNS} --data-root {DATA} --out /kaggle/working/llm --workers 3"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for qudar (pyproject.toml) ... done


In [3]:
# Dùng lại bộ nhớ tạm của các lần chạy trước, để chế độ paper trên HotpotQA 1.500 query không tốn tiền
!mkdir -p /kaggle/working/llm && cp $(find /kaggle/input -path "*qudar-llm*" -name "llm_*_test.jsonl") /kaggle/working/llm/
!ls /kaggle/working/llm

# Chế độ paper trên tập con HotpotQA: lấy từ bộ nhớ tạm, 0 USD
!{L} --mode paper --names hotpotqa_test --max-queries 1500

# 3 chế độ còn thiếu
for m in ["code", "paper_fixed", "code_shuffle"]:
    !{L} --mode {m} --names scidocs_test climate-fever_test
    !{L} --mode {m} --names hotpotqa_test --max-queries 1500

llm_code_fiqa_test.jsonl	  llm_paper_climate-fever_test.jsonl
llm_dat_climate-fever_test.jsonl  llm_paper_fiqa_test.jsonl
llm_dat_fiqa_test.jsonl		  llm_paper_hotpotqa_test.jsonl
llm_dat_hotpotqa_test.jsonl	  llm_paper_scidocs_test.jsonl
llm_dat_scidocs_test.jsonl
hotpotqa_test [paper]: 1500 queries, 7405 cached, 0 to score
hotpotqa_test: 0 failed (equal weights used), avg tokens/query 501
LLM fusion: 100%|██████████████████████████| 1500/1500 [00:01<00:00, 862.44it/s]
                   name    method  Recall@10  nDCG@10  Precision@1
hotpotqa_test_first1500 qudar_llm     0.6737   0.6366       0.7113
scidocs_test [code]: 1000 queries, 0 cached, 1000 to score
  500/1000 scored, 253/min, ~2 min left
  1000/1000 scored, 257/min, ~0 min left
scidocs_test: 0 failed (equal weights used), avg tokens/query 811
LLM fusion: 100%|██████████████████████████| 1000/1000 [00:01<00:00, 817.90it/s]
        name               method  Recall@10  nDCG@10  Precision@1
scidocs_test qudar_llm_codeprompt     